In [ ]:
import os, time, json
from pathlib import Path
import torch
import torch_xla
import torch_xla.core.xla_model as xm
import torch_xla.runtime as xr
print('torch', torch.__version__, 'xla', torch_xla.__version__)
print('world_size', xr.world_size(), 'ordinal', xr.global_ordinal())
dev = xm.xla_device(); print('device', dev)


In [ ]:
import kagglehub
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
p = kagglehub.model_download('qwen-lm/qwen-3/Transformers/4b/1')
print('model', p)
tok = AutoTokenizer.from_pretrained(p, trust_remote_code=True)
if tok.pad_token is None: tok.pad_token = tok.eos_token
base = AutoModelForCausalLM.from_pretrained(p, dtype=torch.bfloat16, trust_remote_code=True, low_cpu_mem_usage=True)
base.gradient_checkpointing_enable(); base.enable_input_require_grads()
model = get_peft_model(base, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM', target_modules='all-linear')).to(dev)
print('loaded, mem', xm.get_memory_info(dev))


In [ ]:
text='Áo thun nam cotton cao cấp'; tgt='男士高档棉质T恤'
ids=tok(text, return_tensors='pt').input_ids.to(dev); lab=ids.clone()
opt=torch.optim.AdamW([q for q in model.parameters() if q.requires_grad], lr=1e-4)
model.train(); t0=time.time()
for i in range(3):
    loss=model(input_ids=ids, labels=lab).loss
    loss.backward(); xm.optimizer_step(opt); xm.mark_step()
    print('step',i,'loss',float(loss.item()),'sec',round(time.time()-t0,2),'mem',xm.get_memory_info(dev), flush=True)
print('OK')
